[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShintaroMinami/notebook_proteina_complexa/blob/main/proteina_complexa.ipynb)

# Proteina Complexa — Hands-On Notebook

このノートブックでは、NVIDIAが開発したタンパク質デザインモデル **Proteina Complexa** を
Google Colab上で実行します。

> **必要なもの:** Googleアカウントのみ。ローカル環境の構築は不要です。

In [ ]:
#@title **ノートブックの初期設定**
import subprocess, shlex
from IPython.display import display, HTML

def run(cmd, title=""):
    """Run a shell command and display output in a collapsible block."""
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    output = (result.stdout + result.stderr).strip()
    lines = output.split("\n")
    icon = "\u2705" if result.returncode == 0 else "\u274c"
    summary = f"{icon} {title} ({len(lines)} lines)" if title else f"{icon} Output ({len(lines)} lines)"
    html = (
        f'<details><summary style="cursor:pointer;font-weight:bold;padding:4px 0">{summary}</summary>'
        f'<pre style="max-height:300px;overflow-y:auto;background:#f5f5f5;padding:8px;'
        f'font-size:11px;border-radius:4px;margin-top:4px">{output}</pre>'
        f'</details>'
    )
    display(HTML(html))
    return result.returncode

print("Ready.")

---
## 1. 環境構築

はじめに、必要なソフトウェアをインストールします。
以下のセルを **上から順に** 実行してください（各セルで Shift+Enter）。

### 1.1 GPU の確認

まず、GPUが割り当てられているか確認します。
「GPU が見つかりません」と表示された場合は、メニューの **「ランタイム」→「ランタイムのタイプを変更」** から
**GPU** を選択してください。

In [ ]:
#@title **GPU の確認**
import subprocess, sys

result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                        capture_output=True, text=True)
if result.returncode == 0:
    gpu_info = result.stdout.strip()
    print(f"GPU: {gpu_info}")
else:
    print("GPU が見つかりません。ランタイムのタイプを変更してください。")

print(f"Python: {sys.version}")

### 1.2 パッケージマネージャ (uv) のインストールとリポジトリの取得

高速パッケージマネージャ **uv** をインストールし、Proteina Complexa のソースコードを取得します。

In [ ]:
#@title **uv のインストールとリポジトリの取得**
run("pip install -q uv", "uv install")
run("git clone --depth 1 https://github.com/NVIDIA-Digital-Bio/Proteina-Complexa.git", "git clone")
import os
os.chdir("Proteina-Complexa")
print(f"Working directory: {os.getcwd()}")

### 1.3 PyTorch のインストール

GPU対応版の PyTorch をインストールします。（数分かかります）

In [ ]:
#@title **PyTorch のインストール**
run(
    "uv pip install --system torch==2.7.0+cu126 torchvision torchaudio"
    " --index-url https://download.pytorch.org/whl/cu126",
    "PyTorch + CUDA 12.6"
)

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

### 1.4 Proteina Complexa のインストール

本体と依存ライブラリをインストールします。（数分かかります）

In [ ]:
#@title **Proteina Complexa のインストール**
run("uv pip install --system --index-strategy unsafe-best-match -e .", "Base package")
run(
    "uv pip install --system torch_geometric torch_scatter torch_sparse torch_cluster"
    " -f https://data.pyg.org/whl/torch-2.7.0+cu126.html",
    "PyTorch Geometric"
)
run("uv pip install --system graphein==1.7.7 --no-deps", "Graphein")
run("uv pip install --system \"atomworks[ml,openbabel,dev]\" 2>/dev/null || true", "Atomworks")
run("uv pip install --system biotite==1.6.0", "biotite")
print("Done.")

### 1.5 構造予測モデル (Boltz-2) のインストール

生成したタンパク質の構造を予測・検証するためのモデルをインストールします。

In [ ]:
#@title **構造予測モデル (Boltz-2) のインストール**
run("uv pip install --system --index-strategy unsafe-best-match \"boltz[cuda]\" -U", "Boltz-2")
print("Done.")

### 1.6 モデルの重みをダウンロード

事前学習済みモデルをダウンロードします。（数分かかります）

In [ ]:
#@title **モデルの重みをダウンロード**
run("complexa init", "complexa init")
run("complexa download --complexa-all", "complexa download")

### 1.7 インストールの確認

全てのインストールが完了したか確認します。

In [ ]:
#@title **インストールの確認**
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

!complexa --help > /dev/null 2>&1 && echo "Proteina Complexa CLI: OK" || echo "Proteina Complexa CLI: NOT FOUND"

print()
print("Environment setup complete!")